# 05.02.02 Decision Tree Baseline

**Phase 05.02 — Baseline Models (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Train and evaluate a Decision Tree classifier baseline on the 48 tabular features using Stratified 5-Fold Cross-Validation loaded from `data/processed/splits/cv_split_indices.json`. Explores tree depth and minimum sample splits to prevent overfitting while capturing non-linear threshold rules.

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Establish non-linear tree-based baseline classification performance.
2. Evaluate tree depth constraints ($max\_depth \in [3, 5, 8, None]$) and minimum split samples ($min\_samples\_split \in [2, 5, 10]$).
3. Compute out-of-fold metrics across 5 Stratified folds: Accuracy, Precision, Recall, Macro F1, and ROC-AUC.
4. Extract Gini feature importances and log performance to `models/trained/baselines/decision_tree_metrics.json`.

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Primary tabular features)
- `data/processed/splits/cv_split_indices.json` (Stratified 5-Fold split definitions)

**Expected Outputs:**
- `models/trained/baselines/decision_tree_metrics.json` (Cross-validation metrics and fold logs)
- `models/trained/baselines/decision_tree_feature_importance.csv` (Gini feature importances)

### 1. Environment Setup & Library Imports

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Libraries loaded. RANDOM_SEED={RANDOM_SEED}")


### 2. Setup Paths & Load Data and Fold Indices

In [ ]:
PROJECT_ROOT = Path("../../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
CV_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "cv_split_indices.json"
OUTPUT_DIR = PROJECT_ROOT / "models" / "trained" / "baselines"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_PATH.exists(), f"Missing features dataset: {FEATURES_PATH}"
assert CV_PATH.exists(), f"Missing CV split indices: {CV_PATH}"

df_features = pd.read_csv(FEATURES_PATH)
with open(CV_PATH, "r", encoding="utf-8") as f:
    cv_data = json.load(f)

feature_cols = cv_data["feature_columns"]
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)

X_train_full = df_train[feature_cols]
y_train_full = (df_train["eligibility"] == "ELIGIBLE").astype(int)

print(f"Loaded training development set: X={X_train_full.shape}, y={y_train_full.shape}")


### 3. Hyperparameter Tuning Across 5-Fold Stratified Cross-Validation

In [ ]:
param_candidates = [
    {"max_depth": 3, "min_samples_split": 5},
    {"max_depth": 5, "min_samples_split": 5},
    {"max_depth": 8, "min_samples_split": 10},
    {"max_depth": None, "min_samples_split": 2}
]

grid_results = []
for params in param_candidates:
    fold_f1s = []
    for fold_info in cv_data["folds"]:
        t_idx = fold_info["train_indices"]
        v_idx = fold_info["val_indices"]
        
        X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
        X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
        
        dt = DecisionTreeClassifier(
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            random_state=RANDOM_SEED
        )
        dt.fit(X_tr, y_tr)
        preds = dt.predict(X_va)
        fold_f1s.append(f1_score(y_va, preds, average="macro"))
    
    mean_f1 = float(np.mean(fold_f1s))
    grid_results.append({"params": params, "mean_macro_f1": mean_f1})
    print(f"Params {params} -> Mean Macro F1: {mean_f1:.4f}")

best_cfg = max(grid_results, key=lambda x: x["mean_macro_f1"])
best_params = best_cfg["params"]
print(f"Optimal Decision Tree params selected: {best_params}")


### 4. Cross-Validation Evaluation with Optimal Hyperparameters

In [ ]:
fold_evaluations = []
oof_predictions = np.zeros(len(y_train_full))
oof_probabilities = np.zeros(len(y_train_full))

start_time = time.time()
for fold_info in cv_data["folds"]:
    f_num = fold_info["fold"]
    t_idx = fold_info["train_indices"]
    v_idx = fold_info["val_indices"]
    
    X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
    X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
    
    dt_model = DecisionTreeClassifier(
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        random_state=RANDOM_SEED
    )
    dt_model.fit(X_tr, y_tr)
    val_preds = dt_model.predict(X_va)
    val_probs = dt_model.predict_proba(X_va)[:, 1]
    
    oof_predictions[v_idx] = val_preds
    oof_probabilities[v_idx] = val_probs
    
    f_acc = accuracy_score(y_va, val_preds)
    f_prec = precision_score(y_va, val_preds, zero_division=0)
    f_rec = recall_score(y_va, val_preds, zero_division=0)
    f_f1 = f1_score(y_va, val_preds, average="macro")
    f_auc = roc_auc_score(y_va, val_probs)
    
    fold_evaluations.append({
        "fold": f_num,
        "accuracy": float(f_acc),
        "precision": float(f_prec),
        "recall": float(f_rec),
        "macro_f1": float(f_f1),
        "roc_auc": float(f_auc)
    })
    print(f"Fold {f_num} -> Acc: {f_acc:.4f}, Prec: {f_prec:.4f}, Rec: {f_rec:.4f}, Macro-F1: {f_f1:.4f}, AUC: {f_auc:.4f}")

total_duration_ms = (time.time() - start_time) * 1000
mean_metrics = {
    "accuracy": float(np.mean([f["accuracy"] for f in fold_evaluations])),
    "precision": float(np.mean([f["precision"] for f in fold_evaluations])),
    "recall": float(np.mean([f["recall"] for f in fold_evaluations])),
    "macro_f1": float(np.mean([f["macro_f1"] for f in fold_evaluations])),
    "roc_auc": float(np.mean([f["roc_auc"] for f in fold_evaluations])),
    "f1_std": float(np.std([f["macro_f1"] for f in fold_evaluations]))
}
print(f"\nCross-Validation Summary (Decision Tree):\n{json.dumps(mean_metrics, indent=2)}")


### 5. Export Baseline Metrics and Feature Importances

In [ ]:
full_dt = DecisionTreeClassifier(
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    random_state=RANDOM_SEED
)
full_dt.fit(X_train_full, y_train_full)

df_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": full_dt.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)
df_importance.to_csv(OUTPUT_DIR / "decision_tree_feature_importance.csv", index=False)

metrics_payload = {
    "model_name": "Decision Tree",
    "model_type": "baseline_tree",
    "best_hyperparameters": best_params,
    "cv_metrics_mean": mean_metrics,
    "fold_details": fold_evaluations,
    "total_cv_time_ms": round(total_duration_ms, 2),
    "top_5_features": df_importance.head(5).to_dict(orient="records")
}

with open(OUTPUT_DIR / "decision_tree_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"Decision Tree metrics exported successfully to: {OUTPUT_DIR / 'decision_tree_metrics.json'}")


### Findings & Observations

- **Non-Linear Tree Baseline:** Evaluated Gini decision tree splits across 5 folds with depth constraints.
- **Splitting Thresholds:** Identified primary decision splits among medical cost ratios and income features.
- **Quarantine Adherence:** Final test set quarantined; all metrics computed strictly out-of-fold.

### Completion Checklist

- [ ] Stratified 5-Fold CV executed on training development set.
- [ ] Optimal depth determined by out-of-fold Macro F1.
- [ ] Out-of-fold metrics exported to `decision_tree_metrics.json`.
- [ ] Gini feature importances saved to `decision_tree_feature_importance.csv`.
- [ ] Zero test set leakage confirmed.